# 02 — Silver Layer
Read Bronze Parquets. Apply all cleaning, privacy masking, and joins.
Build the Star Schema dimension tables and fact table.
Save Silver Parquet and all dimension/fact Parquets.

## Load Bronze Parquets

In [1]:
import pandas as pd
import numpy as np
import hashlib

tickets   = pd.read_parquet("bronze_tickets.parquet")
customers = pd.read_parquet("bronze_customers.parquet")

print("Tickets: ", tickets.shape)
print("Customers:", customers.shape)

Tickets:  (9119, 20)
Customers: (10000, 10)


## Mask PII — done once, here only
Customer name, contact person, email and address hashed with SHA-256 before any analysis.

In [3]:
def hash_value(val):
    if pd.isna(val):
        return val
    return hashlib.sha256(str(val).encode()).hexdigest()[:16]

pii_columns = ["customer_name", "contact_person", "email_address", "Address_line_1"]

for col in pii_columns:
    if col in customers.columns:
        customers[col] = customers[col].apply(hash_value)

print("PII masked:", pii_columns)
customers[pii_columns].head()

PII masked: ['customer_name', 'contact_person', 'email_address', 'Address_line_1']


,customer_name,contact_person,email_address,Address_line_1
0,01332c876518a793,None,c4d25e9c90ff23e9,2e58196c7c240eca
1,6cea57c2fb6cbc2a,None,836f82db99121b34,948595bd0490ff95
2,None,db8db1c8e00fd626,4a084e90c098ac98,d8c1c4659640b640
3,None,3e5f2809db48e9c4,4c2b54287c2fa921,65c8bea6303617b8
4,None,6cea57c2fb6cbc2a,a41cd188c7ed3023,8b2e7fba8625e03a


## Join customer info onto tickets

In [4]:
df = tickets.merge(
    customers[["customer_id", "country", "city"]],
    on="customer_id", how="left"
)

print("Rows after join:", len(df))
df[["ticket_id", "customer_id", "country", "city", "region"]].head()

Rows after join: 9279


,ticket_id,customer_id,country,city,region
0,TCKT_000001,CUST_00861,Germany,Berlin,EU
1,TCKT_000002,CUST_00770,United States,New York,None
2,TCKT_000003,CUST_02559,United Arab Emirates,Abu Dhabi,MEA
3,TCKT_000004,CUST_03557,Brazil,São Paulo,LATAM
4,TCKT_000005,CUST_09556,United States,Beverly Hills,None


## Fill missing region values from country
~20,000 region values are missing. Backfilled using a country-to-region lookup. Remaining nulls dropped.

In [5]:
country_to_region = {
    "United States": "NA",  "Canada": "NA",       "Mexico": "NA",
    "Germany": "EU",        "France": "EU",        "United Kingdom": "EU",
    "Spain": "EU",          "Italy": "EU",         "Netherlands": "EU",
    "Sweden": "EU",         "Poland": "EU",        "Belgium": "EU",
    "China": "APAC",        "Japan": "APAC",       "India": "APAC",
    "Australia": "APAC",    "South Korea": "APAC", "Singapore": "APAC",
    "Brazil": "LATAM",      "Argentina": "LATAM",  "Colombia": "LATAM",
    "Chile": "LATAM",       "Peru": "LATAM",
    "South Africa": "MEA",  "Nigeria": "MEA",      "Kenya": "MEA",
    "Egypt": "MEA",         "Saudi Arabia": "MEA", "UAE": "MEA",
}

missing_before = df["region"].isna().sum()
mask = df["region"].isna()
df.loc[mask, "region"] = df.loc[mask, "country"].map(country_to_region)
df = df.dropna(subset=["region"])

print(f"Missing region before: {missing_before}")
print(f"Missing region after:  {df['region'].isna().sum()}")
print(f"Rows remaining: {len(df)}")

Missing region before: 1824
Missing region after:  0
Rows remaining: 7817


## Recode CSAT 0 → null
The survey scale is 1–5. A score of 0 means no response — not a real rating. Recoded to null before any CSAT calculation.

In [6]:
print("Before:", df["csat_score"].value_counts().sort_index().to_dict())
df["csat_score"] = df["csat_score"].replace(0, np.nan)
print("After: ", df["csat_score"].value_counts(dropna=False).sort_index().to_dict())

Before: {0: 2342, 1: 461, 2: 1173, 3: 1549, 4: 1361, 5: 931}
After:  {1.0: 461, 2.0: 1173, 3.0: 1549, 4.0: 1361, 5.0: 931, nan: 2342}


## Add derived columns

In [7]:
df["year_month"]  = df["created_at"].dt.to_period("M").astype(str)
df["is_resolved"] = df["status"].isin(["resolved", "closed_no_action"])

print("Sample:")
df[["ticket_id", "year_month", "is_resolved"]].head()

Sample:


,ticket_id,year_month,is_resolved
0,TCKT_000001,2024-01,True
1,TCKT_000002,2024-10,True
2,TCKT_000003,2024-06,False
3,TCKT_000004,2025-12,False
4,TCKT_000005,2023-08,True


## Final null check

In [8]:
print("Total rows:", len(df))
print()
print("Remaining nulls (expected: resolution cols and csat for open/no-response tickets):")
print(df.isnull().sum()[df.isnull().sum() > 0])

Total rows: 7817

Remaining nulls (expected: resolution cols and csat for open/no-response tickets):
resolution_summary       3150
resolution_time_hours    3150
csat_score               2342
country                  5771
city                     5771
dtype: int64


## Save Silver Parquet

In [9]:
df.to_parquet("silver.parquet", index=False)
print("Saved: silver.parquet —", len(df), "rows")

Saved: silver.parquet — 7817 rows


## Build Star Schema
The report uses a Star Schema:
- **fact_tickets** — one row per ticket, numeric measures and foreign keys
- **dim_customer** — one row per customer: segment, SLA plan, geography
- **dim_ticket_type** — one row per unique combination of issue type, product area, channel, platform
- **dim_date** — one row per day: year, month, quarter, day of week


In [11]:
dim_customer = (
    df[["customer_id", "customer_segment", "sla_plan", "country", "city", "region"]]
    .drop_duplicates(subset=["customer_id"])
    .reset_index(drop=True)
)
dim_customer.columns = ["customer_id", "segment", "sla_plan", "country", "city", "region"]
print("dim_customer:", dim_customer.shape)
dim_customer.head()

dim_customer: (5296, 6)


,customer_id,segment,sla_plan,country,city,region
0,CUST_00861,individual,standard,Germany,Berlin,EU
1,CUST_00770,individual,standard,United States,New York,NA
2,CUST_02559,small_business,standard,United Arab Emirates,Abu Dhabi,MEA
3,CUST_03557,education,standard,Brazil,São Paulo,LATAM
4,CUST_09556,enterprise,gold,United States,Beverly Hills,NA


In [12]:
dim_ticket_type = (
    df[["issue_type", "product_area", "channel", "platform"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_ticket_type.insert(0, "ticket_type_id", range(1, len(dim_ticket_type) + 1))
print("dim_ticket_type:", dim_ticket_type.shape)
dim_ticket_type.head()

dim_ticket_type: (1398, 5)


,ticket_type_id,issue_type,product_area,channel,platform
0,1,account_access,data_export,email,android
1,2,security_concern,billing,in_app,web
2,3,bug,api_integration,chat,android
3,4,account_access,analytics_dashboard,chat,android
4,5,billing_problem,login_auth,phone_transcript,web


In [13]:
dates = pd.DataFrame({"date": pd.to_datetime(df["created_at"].dt.date.unique())})
dates = dates.sort_values("date").reset_index(drop=True)
dates["date_key"]    = dates["date"].dt.strftime("%Y%m%d").astype(int)
dates["year"]        = dates["date"].dt.year
dates["month"]       = dates["date"].dt.month
dates["quarter"]     = dates["date"].dt.quarter
dates["day_of_week"] = dates["date"].dt.day_name()
dim_date = dates[["date_key", "date", "year", "month", "quarter", "day_of_week"]]
print("dim_date:", dim_date.shape)
dim_date.head()

dim_date: (1456, 6)


,date_key,date,year,month,quarter,day_of_week
0,20220101,2022-01-01,2022,1,1,Saturday
1,20220102,2022-01-02,2022,1,1,Sunday
2,20220103,2022-01-03,2022,1,1,Monday
3,20220104,2022-01-04,2022,1,1,Tuesday
4,20220105,2022-01-05,2022,1,1,Wednesday


In [14]:

df_fact = df.merge(
    dim_ticket_type[["ticket_type_id", "issue_type", "product_area", "channel", "platform"]],
    on=["issue_type", "product_area", "channel", "platform"],
    how="left"
)

fact_tickets = df_fact[[
    "ticket_id", "customer_id", "ticket_type_id",
    "created_at", "status", "priority",
    "resolution_time_hours", "csat_score",
    "reopened", "has_attachment",
    "year_month", "is_resolved"
]].copy()

print("fact_tickets:", fact_tickets.shape)
fact_tickets.head()

fact_tickets: (7817, 12)


,ticket_id,customer_id,ticket_type_id,created_at,status,priority,resolution_time_hours,csat_score,reopened,has_attachment,year_month,is_resolved
0,TCKT_000001,CUST_00861,1,2024-01-31 05:14:27,resolved,low,36.53,1.0,0,0,2024-01,True
1,TCKT_000002,CUST_00770,2,2024-10-20 06:15:49,closed_no_action,medium,238.32,3.0,0,0,2024-10,True
2,TCKT_000003,CUST_02559,3,2024-06-18 21:35:54,in_progress,low,NaN,3.0,0,0,2024-06,False
3,TCKT_000004,CUST_03557,4,2025-12-25 15:59:52,in_progress,medium,NaN,5.0,0,1,2025-12,False
4,TCKT_000005,CUST_09556,5,2023-08-27 16:08:33,resolved,low,61.32,2.0,0,0,2023-08,True


## Save Star Schema Parquets

In [15]:
dim_customer.to_parquet("dim_customer.parquet",       index=False)
dim_ticket_type.to_parquet("dim_ticket_type.parquet", index=False)
dim_date.to_parquet("dim_date.parquet",               index=False)
fact_tickets.to_parquet("fact_tickets.parquet",       index=False)

print("Saved Star Schema:")
print(f"  dim_customer.parquet    — {len(dim_customer):,} rows")
print(f"  dim_ticket_type.parquet — {len(dim_ticket_type):,} rows")
print(f"  dim_date.parquet        — {len(dim_date):,} rows")
print(f"  fact_tickets.parquet    — {len(fact_tickets):,} rows")

Saved Star Schema:
  dim_customer.parquet    — 5,296 rows
  dim_ticket_type.parquet — 1,398 rows
  dim_date.parquet        — 1,456 rows
  fact_tickets.parquet    — 7,817 rows
